# AI / ML: Deep Learning – GenAI - Transformers
- Text Summarizer App (HuggingFace Transformer)

In [1]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration
import re
import torch

## Reading Data:

In [2]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [3]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
val_data.head()

,id,dialogue,summary
0,13817023,"A: Hi Tom, are you busy tomorrow’s afternoon?\...",A will go to the animal shelter tomorrow to ge...
1,13716628,Emma: I’ve just fallen in love with this adven...,Emma and Rob love the advent calendar. Lauren ...
2,13829420,Jackie: Madison is pregnant\r\nJackie: but she...,Madison is pregnant but she doesn't want to ta...
3,13819648,Marla: <file_photo>\r\nMarla: look what I foun...,Marla found a pair of boxers under her bed.
4,13728448,Robert: Hey give me the address of this music ...,Robert wants Fred to send him the address of t...


In [5]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [6]:
train_data.sample(10)

,id,dialogue,summary
11420,13828527,Jane: Hi Ted... Danny told me you'll come tomo...,"Ted hasn't been to school in two months, which..."
3802,13716255,Dakota: you won't believe me but I've just wat...,Dakota quite liked 50 shades of Grey. Stephani...
11064,13811380,Abigail: when did you read read their texts? :...,Ethan and Abigail are surprised that she did n...
2994,13611705,Ella: Hi there! Howdy? Haven't heard from you ...,"Ella has a nasty stomach bug, she is in pain. ..."
5891,13681148-1,Pete: Have you seen my wallet? I think I left ...,Pete lost his wallet. Sandy will look for it.
1277,13716200,Mona: Do you need any fruit or veg?\r\nDon: Su...,Suzie wants Mona to buy some apples and pears....
12097,13862312,Donna: Can you help me with this?\nDonna: <fil...,Martha is going to help Donna with this after ...
4996,13728621,"Rachel: Hi, how are you doing?\r\nCharlie: goo...",Charlie is in New York. Rachel will see Charli...
10546,13730247,Gavin: hey u asleep lol \r\nChelsea: about to ...,It's 1 AM and Chelsea is about to go to sleep....
9003,13681035,Ken: <file_photo>\r\nKen: Only you can see thi...,Friends have given Ken a present he had craved...


In [7]:
train_data.shape

(14732, 3)

In [8]:
val_data.shape

(818, 3)

## Random Sampeling:

In [9]:
train_data = train_data.sample(
    n = 4000,
    random_state = 42
).reset_index(drop = True)

val_data = val_data.sample(
    n = 500,
    random_state = 42
).reset_index(drop = True)

In [10]:
train_data.shape

(4000, 3)

In [11]:
val_data.shape

(500, 3)

## Data Preprocessing: Cleaning Data

In [12]:
def clean_data(text):
    text = re.sub(r"\r\n", " ", text) # lines
    text = re.sub(r"\s+", " ", text) # spaces
    text = re.sub(r"\<.*?>", " ", text) # html tags
    text = text.strip().lower()
    return text

In [13]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [14]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

## Tokenization our Data for T5: Model

In [15]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [16]:
# Function to convert row data to tokens:
# Tokenise input for fine-tuning of our Model training.

def tokenize(data):
    inputs = tokenizer( # Input Values
        data["dialogue"],
        padding = "max_length",
        max_length = 512,
        truncation = True
    )

    targets = tokenizer( # Output Values
        data["summary"],
        padding = "max_length",
        max_length = 150,
        truncation = True
    )

    inputs["labels"] = targets["input_ids"] # Token ids => added to input as labels

    return inputs

## Calling the 'tokenize()' function
- For Training & validation Data.

In [17]:
train_dataset = train_data.apply(tokenize, axis = 1).tolist()
val_dataset = val_data.apply(tokenize, axis = 1).tolist()

In [18]:
train_dataset[0]

# input ids - dialogues (token ids) 
# attention mask - inputs id => valid values or padding vals
# labels - target vals (summary)

# 1 - end of sequences
# 0 - added paddings

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [19]:
len(train_dataset[0]["input_ids"])

512

In [20]:
type(train_dataset)

list

In [21]:
type(val_dataset)

list

## Fine-Tuning our Transformer Model:
- Working with our Model.

In [22]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [23]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Device: {device}")

Device: cpu


In [24]:
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [25]:
# Training Arguments:

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 6,
    weight_decay = 0.01,

    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy = "epoch",
    save_strategy = "epoch",

    warmup_steps = 500
)

## Training our Model:

In [26]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

## Model Training: Slowest Part of the Project

In [27]:
# trainer.train()

## Saving & Loading our Model:

In [28]:
# model.save_pretrained("./saved_summary_model")
# tokenizer.save_pretrained("./saved_summary_model")

In [29]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## Testing the Sumarization Logic: Model

In [30]:
def summarization_dialogue(dialogue):
    dialogue = clean_data(dialogue) # Clean dialogue data

    # Tokanizing:
    inputs = tokenizer(
        dialogue,
        padding = "max_length",
        max_length = 512,
        truncation = True,
        return_tensors = "pt"
    )

    # Generating the Summary: Token ID
    model.to(device)
    targets = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length = 150,
        num_beams = 4,
        early_stopping = True
    )
 
    # Converting Token IDs to Text: Decoding our output
    summary = tokenizer.decode(
        targets[0],
        skip_special_tokens = True,
    )

    return summary


## Priving Sample Inputs for our Summary function: Testing...

In [31]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks like language translation and medical image analysis.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is a major challenge.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and use of AI technologies.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as "black boxes," making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, accountability, and ethical considerations.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be essential to ensure AI benefits society while minimizing risks.
"""

In [32]:
summary = summarization_dialogue(test_dialogue)
print(f"Summary: {summary}")

Summary: , companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. experts highlight the importance of responsible ai development, including data privacy, accountability, and ethical considerations.
